In [0]:
### BUSINESS PROFILE

#### AxisOne Bank is a leading private sector bank with over 28 years of operations across India. The bank serves more than 12 million retail customers through a network of 3,200 branches, digital banking platforms, mobile applications, and relationship managers. Every day, millions of customer transactions flow through the bank's core banking systems, internet banking portals, loan management systems, customer relationship management (CRM) platform, KYC systems, fraud detection engines, and credit risk platforms.

### PROBLEM STATEMENT

### To support business intelligence, regulatory reporting, customer analytics, and risk management, AxisOne Bank has established a centralized enterprise data platform built on Delta Lake. Data from fourteen operational systems is ingested into the platform through scheduled ETL and ELT pipelines, where it is transformed into trusted datasets for downstream consumers.

## One of the most business-critical assets within this platform is the CUSTOMER DIMENSION, which serves as the single source of truth for customer information. This dimension is consumed by several downstream applications, including loan approval systems, anti-money laundering (AML) monitoring, fraud analytics, regulatory reporting, customer segmentation, marketing analytics, and executive dashboards.

## Because the banking industry is heavily regulated by the Reserve Bank of India (RBI), every modification to a customer's profile must be historically traceable. Changes such as customer address updates, KYC status modifications, mobile number changes, email updates, and risk segment revisions must never overwrite historical values. Instead, every version of a customer's record must remain available for future audits and investigations.

## Maintaining complete historical lineage is essential to demonstrate regulatory compliance, reproduce historical business decisions, investigate customer disputes, and support accurate analytical reporting.

In [0]:
### Problem Statement

### During the annual RBI audit, the regulatory team requested historical customer information to validate several loan approvals issued over the previous year.

### Specifically, auditors asked the bank to determine:

## What was the customer's Risk Segment on the day the loan was approved?
### Was the customer's KYC verification completed before the loan was sanctioned?
## Which mobile number and email address were registered at the time of approval?
## Which customer address was active when the loan was disbursed?
## When exactly did each customer attribute change?
## Who initiated the change?
## Can the bank reproduce historical customer records without relying on external backups?

### The audit team quickly discovered that the Customer Dimension only contained the latest values because historical records had been overwritten during every daily load. Consequently, the bank could not reproduce historical customer information for previous dates.

In [0]:
#### You have joined the Enterprise Data Engineering team as the Senior Data Engineer responsible for redesigning the Customer Dimension.

### Your responsibility is to build a production-ready SCD Type 2 pipeline using Apache Spark and Delta Lake that supports complete historical tracking while maintaining scalability, reliability, and regulatory compliance.The solution must be capable of processing daily customer snapshots, identifying inserts, updates, unchanged records, and invalid records before safely merging them into the Customer Dimension.

## The implementation must also guarantee idempotent processing so that rerunning the same batch does not introduce duplicate historical records.Finally, the solution should enable auditors to retrieve customer information exactly as it existed on any historical date using Delta Lake versioning and effective date ranges.

In [0]:
#### Initializing the spark session
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.types import*
spark= SparkSession.builder.appName("vyomnet").getOrCreate()

In [0]:
### Reading the datasets from the volume and building the dataframes

audit_log_df= spark.read.format("csv").option("header","true").option("mode","PERMISSIVE").option("inferSchema","false").load("/Volumes/coding_playbook/4_axisonebank/raw_data/audit_log.csv")

branch_master_df= spark.read.format("csv").option("header","true").option("mode","PERMISSIVE").option("inferSchema","false").load("/Volumes/coding_playbook/4_axisonebank/raw_data/branch_master.csvv")

contact_history_df= spark.read.format("csv").option("header","true").option("mode","PERMISSIVE").option("inferSchema","false").load("/Volumes/coding_playbook/4_axisonebank/raw_data/customer_contact_history.csv")

kyc_verification_df= spark.read.format("csv").option("header","true").option("mode","PERMISSIVE").option("inferSchema","false").load("/Volumes/coding_playbook/4_axisonebank/raw_data/kyc_verification.csv")

In [0]:
##### For building the SCD Type 2 Merge Logic we have created one sample dataframe with just one entry that is supposed to have the final data 

sample_data = [("CUST102199", "9410776250", "5440681847", "2025-06-08","9999-06-08","Yes")]

columns = ["customer_id","old_mobile","new_mobile","start_date","end_date","is_active"]

final_customer_history_df= spark.createDataFrame(sample_data,columns)

final_customer_history_df.write.format("delta").mode("overwrite").save("/Volumes/coding_playbook/4_axisonebank/final_data/final_customer_data/")

final_customer_history_df= spark.read.format("delta").load("/Volumes/coding_playbook/4_axisonebank/final_data/final_customer_data/")

final_customer_history_df.show()

#### Also we have to store our customer history table in Delta format
contact_history_df.write.format("delta").save("/Volumes/coding_playbook/4_axisonebank/final_data/customer_delta_table/")

In [0]:
##### SCD TYPE 2 MERGE LOGIC

watermark_date=watermark_date = contact_history_df.agg(f.max("change_date").alias("watermark_date")).collect()[0][0]

contact_history_df_req= spark.read.format("delta").load("/Volumes/coding_playbook/4_axisonebank/final_data/customer_delta_table/")

### Firstly we have to expire th old records and then insert the brand new records
def scd_type_2_merge(final_customer_history_df,contact_history_df_req):
    final_customer_history_df=final_customer_history_df_req.alias("tgt").merge(contact_history_df.alias("src"),condition=((f.col("tgt.customer_id")==f.col("src.customer_id"))&(f.col("tgt.is_active")==f.lit("Yes")))).whenMatchedUpdate(set={"tgt.end_date":"src.change_date","tgt.is_active":"No"}).whenNotMatchedInsert(values={"customer_id":"src.customer_id","old_mobile":"src.old_mobile","new_mobile":"src.new_mobile","start_date":"src.change_date", "end_date":f.lit("9999-01-01"),"is_active":"Yes"}).execute()

### Next step is about findign the changed records and appending them so that the new+changed records along with the history are maintained

changed_records= contact_history_df_req.alias("src").join(final_customer_history_df.alias("tgt"),on=((f.col("src.customer_id")==f.col("tgt.customer_id"))&((f.col("src.old_mobile")!=f.col("tgt.old_mobile"))| ((f.col("src.new_mobile")!=f.col("tgt.new_mobile")))))).select(f.col("src.customer_id"),f.col("src.old_mobile"),f.col("src.new_mobile"),f.col("tgt.start_date"),f.col("tgt.end_date"),f.col("tgt.is_active"))

changed_records.write.format("delta").mode("append").save("/Volumes/coding_playbook/4_axisonebank/final_data/final_customer_data/")